In [27]:
from pathlib import Path
import sys
import importlib

# Support kernels launched from the notebook, repository, or workspace directory.
cwd = Path.cwd()
module_dir = next(
    (path for path in (cwd, cwd / "sara", cwd / "hackathon-2026" / "sara")
     if (path / "recommender.py").is_file()),
    None,
)
if module_dir is None:
    raise RuntimeError("Open this notebook with a kernel in the notebook or project directory.")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

# Reload local modules so rerunning this cell picks up changes without stale data.
import recommender
import mock_data
import recommendations
importlib.reload(recommender)
importlib.reload(mock_data)
importlib.reload(recommendations)
from mock_data import AS_OF, make_mock_data
from recommender import EventRecommender

users, events, interactions = make_mock_data()
model = EventRecommender(users, events, interactions, as_of=AS_OF)
assert all(not user.interests for user in users)  # No survey data.
print(f"{len(users)} users, {len(events)} events, {len(interactions)} interactions")
print(f"Demo clock: {AS_OF.isoformat()}")

12 users, 38 events, 50 interactions
Demo clock: 2026-10-03T12:00:00+00:00


In [28]:
def show_recommendations(recommender, user_id, **options):
    # Match the frontend function's relevance policy and default top-12 limit.
    options.setdefault("limit", 12)
    options.setdefault("exclude_seen", False)
    options.setdefault("relevant_only", True)
    print(f"\nRecommendations for {recommender.users[user_id].name}")
    for rank, recommendation in enumerate(recommender.recommend(user_id, **options), 1):
        event = recommendation.event
        print(f"{rank}. {event.title} | {event.starts_at:%Y-%m-%d} | {event.city}")
        print(
            f"   score={recommendation.score:.3f}; "
            f"similar users={recommendation.collaborative_score:.3f}, "
            f"history/tag match={recommendation.content_score:.3f}, "
            f"popularity={recommendation.popularity_score:.3f}"
        )
        for reason in recommendation.reasons:
            print(f"   - {reason}")

# Alice scrolled 100% through an AI event page and 10% through a Python page.
# Her profile comes from those events' tags, not a survey.
show_recommendations(model, "alice")


Recommendations for Alice
1. Hands-on AI lab | 2026-10-10 | Warsaw
   score=0.839; similar users=0.641, history/tag match=0.883, popularity=0.924
   - 2 people with similar event history also viewed or saved this event
   - Saved by people with similar event history
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
2. Machine learning workshop | 2026-10-22 | Warsaw
   score=0.708; similar users=0.269, history/tag match=0.883, popularity=0.364
   - 1 person with similar event history also viewed or saved this event
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
3. New AI research evening | 2026-10-21 | Warsaw
   score=0.691; similar users=0.000, history/tag match=0.986, popularity=0.000
   - Matches your interests/history: technology, ai
4. Responsible AI discussion | 2026-10-29 | Warsaw
   score=0.691; similar users=0.000, history/tag match=0.986, popularity=0.000
   - Matches your interests/history: te

In [29]:
# Without any interaction history, both new users fall back to popularity.
show_recommendations(model, "new", limit=3)
show_recommendations(model, "new_no_interests", limit=3)

# A new AI event can match Alice's viewed-event tags without its own feedback.
new_event_result = next(
    result for result in model.recommend("alice", limit=100)
    if result.event.id == "new_ai"
)
assert new_event_result.content_score > 0
assert new_event_result.collaborative_score == 0


Recommendations for New user
1. Forest trail run | 2026-10-15 | Warsaw
   score=1.000; similar users=0.000, history/tag match=0.000, popularity=1.000
   - Popular with other users
2. Hands-on AI lab | 2026-10-10 | Warsaw
   score=0.924; similar users=0.000, history/tag match=0.000, popularity=0.924
   - Popular with other users
3. Autumn jazz festival | 2026-10-12 | Warsaw
   score=0.725; similar users=0.000, history/tag match=0.000, popularity=0.725
   - Popular with other users

Recommendations for Another new user
1. Forest trail run | 2026-10-15 | Warsaw
   score=1.000; similar users=0.000, history/tag match=0.000, popularity=1.000
   - Popular with other users
2. Hands-on AI lab | 2026-10-10 | Warsaw
   score=0.924; similar users=0.000, history/tag match=0.000, popularity=0.924
   - Popular with other users
3. Autumn jazz festival | 2026-10-12 | Warsaw
   score=0.725; similar users=0.000, history/tag match=0.000, popularity=0.725
   - Popular with other users


In [30]:
from recommender import Interaction

# Simulate Alice exploring the top recommendation's page all the way to 100%.
new_interaction = Interaction(
    user_id="alice", event_id="ai_lab", scroll_percent=100, occurred_at=AS_OF
)
updated_model = EventRecommender(
    users, events, interactions + [new_interaction], as_of=AS_OF
)
show_recommendations(updated_model, "alice", limit=3, city="Warsaw")
assert updated_model.history["alice"]["ai_lab"] == 8
assert "ai_lab" not in {
    result.event.id for result in updated_model.recommend("alice", limit=100)
}

# Fully scrolling does not mean booking. Allow the page to reappear if desired.
assert "ai_lab" in {
    result.event.id
    for result in updated_model.recommend("alice", limit=100, exclude_seen=False)
}
# In production, persist new_interaction before rebuilding/refreshing the model.


Recommendations for Alice
1. Hands-on AI lab | 2026-10-10 | Warsaw
   score=0.920; similar users=0.778, history/tag match=0.960, popularity=0.924
   - 2 people with similar event history also viewed or saved this event
   - Saved by people with similar event history
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
2. Machine learning workshop | 2026-10-22 | Warsaw
   score=0.771; similar users=0.315, history/tag match=0.960, popularity=0.364
   - 1 person with similar event history also viewed or saved this event
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
3. New AI research evening | 2026-10-21 | Warsaw
   score=0.660; similar users=0.000, history/tag match=0.942, popularity=0.000
   - Matches your interests/history: technology, ai


In [31]:
# Alice can save directly from an event card without any scroll data.
save_interaction = Interaction(
    user_id="alice", event_id="ai_lab", scroll_percent=None,
    occurred_at=AS_OF, action="save",
)
saved_model = EventRecommender(
    users, events, interactions + [new_interaction, save_interaction], as_of=AS_OF
)
assert saved_model.history["alice"]["ai_lab"] == 10  # max(100% scroll=8, save=10)
assert "ai_lab" in saved_model.saved["alice"]
show_recommendations(saved_model, "alice", limit=3)

# Saving is interest, not booking. It can be resurfaced when explicitly enabled.
assert "ai_lab" in {
    result.event.id
    for result in saved_model.recommend("alice", limit=100, exclude_seen=False)
}
# Persist save_interaction when the save action succeeds, then refresh the model.


Recommendations for Alice
1. Hands-on AI lab | 2026-10-10 | Warsaw
   score=0.929; similar users=0.798, history/tag match=0.967, popularity=0.924
   - 2 people with similar event history also viewed or saved this event
   - Saved by people with similar event history
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
2. Machine learning workshop | 2026-10-22 | Warsaw
   score=0.778; similar users=0.321, history/tag match=0.967, popularity=0.364
   - 1 person with similar event history also viewed or saved this event
   - Matches your interests/history: technology, ai, coding
   - Popular with other users
3. New AI research evening | 2026-10-21 | Warsaw
   score=0.653; similar users=0.000, history/tag match=0.933, popularity=0.000
   - Matches your interests/history: technology, ai


In [32]:
from recommendations import get_recommended_event_ids

# Pass the user ID received from the frontend; return ranked event IDs.
get_recommended_event_ids("eve")

['trail_run',
 'stretching',
 'park_run',
 'nordic_walk',
 'charity_run',
 'bootcamp',
 'lake_walk',
 'hill_run',
 'urban_hike',
 'outdoor_yoga',
 'forest_hike',
 'cycling']

In [33]:
# Scroll data lives in Interaction.scroll_percent, built from mock_data.behavior.
for interaction in interactions:
    if interaction.user_id == "eve":
        print(
            f"{interaction.event_id}: scroll={interaction.scroll_percent}%, "
            f"action={interaction.action}, at={interaction.occurred_at.isoformat()}"
        )

# Inspect scores: largest first. The frontend receives IDs in this same order.
show_recommendations(model, "eve")
eve_results = model.recommend("eve", limit=12, exclude_seen=False, relevant_only=True)
assert [r.score for r in eve_results] == sorted((r.score for r in eve_results), reverse=True)

past_hike: scroll=100%, action=view, at=2026-09-25T12:00:00+00:00
trail_run: scroll=60%, action=view, at=2026-10-02T12:00:00+00:00
jazz_festival: scroll=10%, action=view, at=2026-09-30T12:00:00+00:00

Recommendations for Eve
1. Forest trail run | 2026-10-15 | Warsaw
   score=0.930; similar users=0.784, history/tag match=0.992, popularity=0.788
   - 2 people with similar event history also viewed or saved this event
   - Matches your interests/history: fitness, outdoors
   - Popular with other users
2. Outdoor stretching class | 2026-10-26 | Warsaw
   score=0.812; similar users=0.381, history/tag match=0.992, popularity=0.414
   - 1 person with similar event history also viewed or saved this event
   - Matches your interests/history: fitness, outdoors
   - Popular with other users
3. Morning park run | 2026-10-09 | Warsaw
   score=0.805; similar users=0.377, history/tag match=0.992, popularity=0.354
   - 1 person with similar event history also viewed or saved this event
   - Matches yo